In [ ]:
import torch

dim = 6

# 1. 生成偶数序列：[0, 2, 4]
# 这里的 0, 2, 4 其实对应公式里的 2i (i是组的索引)
idx = torch.arange(0, dim, 2).float() 
print("idx:", idx) # 输出: tensor([0., 2., 4.])

# 2. 计算频率 (对应的公式是 10000 ^ (-2i/dim))
# 算出来分别是: 10000^0, 10000^(-2/6), 10000^(-4/6)
inv_freq = 1.0 / (10000 ** (idx / dim))
print("inv_freq:", inv_freq) 
# 输出大概是: tensor([1.0000, 0.0464, 0.0022]) 
# 这就是 3 个小组各自的“旋转速度”。

idx: tensor([0., 2., 4.])
inv_freq: tensor([1.0000, 0.0464, 0.0022])


In [2]:
max_seq_len = 5

# 3. 生成所有可能的位置：[0, 1, 2, 3, 4]
t = torch.arange(max_seq_len).float()
print("t (位置):", t) # 输出: tensor([0., 1., 2., 3., 4.])

# 4. 用“位置”乘以“频率”，算出每个 token 每一组的绝对角度
# 注意：t 形状是 [5]，inv_freq 形状是 [3]
# torch.outer 会把它们两两相乘，变成一个 [5, 3] 的矩阵
freqs = torch.outer(t, inv_freq) 
print("freqs (角度表):\n", freqs)

t (位置): tensor([0., 1., 2., 3., 4.])
freqs (角度表):
 tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 4.6416e-02, 2.1544e-03],
        [2.0000e+00, 9.2832e-02, 4.3089e-03],
        [3.0000e+00, 1.3925e-01, 6.4633e-03],
        [4.0000e+00, 1.8566e-01, 8.6177e-03]])


In [12]:
# 5. 算 cos 和 sin
cos_table = freqs.cos()
sin_table = freqs.sin()

# 6. 魔法操作 repeat_interleave
# 作用：把 [A, B, C] 变成 [A, A, B, B, C, C]
cos_table = cos_table.repeat_interleave(2, dim=-1)
sin_table = sin_table.repeat_interleave(2, dim=-1)

print("cos_table 形状:", cos_table.shape) # 输出: [5, 6]
print("cos_table :\n", cos_table)
print("cos_table 形状:", sin_table.shape) # 输出: [5, 6]
print("cos_table :\n", sin_table)

cos_table 形状: torch.Size([5, 6])
cos_table :
 tensor([[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.5403,  0.9989,  0.9989,  1.0000,  1.0000],
        [-0.4161, -0.4161,  0.9957,  0.9957,  1.0000,  1.0000],
        [-0.9900, -0.9900,  0.9903,  0.9903,  1.0000,  1.0000],
        [-0.6536, -0.6536,  0.9828,  0.9828,  1.0000,  1.0000]])
cos_table 形状: torch.Size([5, 6])
cos_table :
 tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.8415,  0.0464,  0.0464,  0.0022,  0.0022],
        [ 0.9093,  0.9093,  0.0927,  0.0927,  0.0043,  0.0043],
        [ 0.1411,  0.1411,  0.1388,  0.1388,  0.0065,  0.0065],
        [-0.7568, -0.7568,  0.1846,  0.1846,  0.0086,  0.0086]])


In [18]:
# 假装我们有一条输入数据，形状是 [1, 5, 6]
x = torch.randn(1, 5, 6)
print('x：',x)
# 7. 奇偶拆分
x_even = x[..., 0::2] # 取第 0, 2, 4 位。形状变成 [1, 5, 3]
print("x_even形状：",x_even.shape)
print("x_even：\n",x_even)
x_odd  = x[..., 1::2] # 取第 1, 3, 5 位。形状变成 [1, 5, 3]
print("x_odd形状：",x_odd.shape)
print("x_odd：\n",x_odd)
# 8. 取出对应的 cos 和 sin（我们也只要偶数位，刚好 3 个一循环）
# 为了能跟 x 匹配，在最前面加个维度，变成 [1, 5, 3]
cos_half = cos_table[..., 0::2].unsqueeze(0)
print("cos_half形状：",cos_half.shape)
print("cos_half：\n",cos_half)

x： tensor([[[ 0.4473, -0.0838,  0.6370,  0.6283, -0.0917,  0.2281],
         [-0.5146, -0.2565,  1.7223, -0.5424,  0.7253,  0.1534],
         [-2.8313, -1.3315,  0.4092, -2.4886, -0.2055, -1.5288],
         [ 0.1849, -0.9294,  0.3674,  0.6178, -0.0443, -1.0836],
         [-0.7563,  0.6635,  0.7509, -0.6707, -0.8742,  0.3144]]])
x_even形状： torch.Size([1, 5, 3])
x_even：
 tensor([[[ 0.4473,  0.6370, -0.0917],
         [-0.5146,  1.7223,  0.7253],
         [-2.8313,  0.4092, -0.2055],
         [ 0.1849,  0.3674, -0.0443],
         [-0.7563,  0.7509, -0.8742]]])
x_odd形状： torch.Size([1, 5, 3])
x_odd：
 tensor([[[-0.0838,  0.6283,  0.2281],
         [-0.2565, -0.5424,  0.1534],
         [-1.3315, -2.4886, -1.5288],
         [-0.9294,  0.6178, -1.0836],
         [ 0.6635, -0.6707,  0.3144]]])
cos_half形状： torch.Size([1, 5, 3])
cos_half：
 tensor([[[ 1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.9989,  1.0000],
         [-0.4161,  0.9957,  1.0000],
         [-0.9900,  0.9903,  1.0000],
         

In [17]:
sin_half = sin_table[..., 0::2].unsqueeze(0)
print("sin_half形状：",sin_half.shape)
print("sin_half：\n",sin_half)

sin_half形状： torch.Size([1, 5, 3])
sin_half：
 tensor([[[ 0.0000,  0.0000,  0.0000],
         [ 0.8415,  0.0464,  0.0022],
         [ 0.9093,  0.0927,  0.0043],
         [ 0.1411,  0.1388,  0.0065],
         [-0.7568,  0.1846,  0.0086]]])


In [19]:
out_even = x_even * cos_half - x_odd * sin_half
print('out_even形状：',out_even.shape)
print('out_even：\n',out_even)

out_even形状： torch.Size([1, 5, 3])
out_even：
 tensor([[[ 0.4473,  0.6370, -0.0917],
         [-0.0622,  1.7456,  0.7250],
         [ 2.3890,  0.6381, -0.1989],
         [-0.0519,  0.2781, -0.0373],
         [ 0.9965,  0.8618, -0.8769]]])


In [20]:
out_odd  = x_even * sin_half + x_odd * cos_half
print('out_odd形状：',out_odd.shape)
print('out_odd：\n',out_odd)

out_odd形状： torch.Size([1, 5, 3])
out_odd：
 tensor([[[-0.0838,  0.6283,  0.2281],
         [-0.5716, -0.4619,  0.1550],
         [-2.0204, -2.4400, -1.5296],
         [ 0.9462,  0.6628, -1.0839],
         [ 0.1387, -0.5206,  0.3068]]])


In [21]:
stacked = torch.stack([out_even, out_odd], dim=-1)
print('stacked形状：',stacked.shape)
print('stacked:\n',stacked)

stacked形状： torch.Size([1, 5, 3, 2])
stacked:
 tensor([[[[ 0.4473, -0.0838],
          [ 0.6370,  0.6283],
          [-0.0917,  0.2281]],

         [[-0.0622, -0.5716],
          [ 1.7456, -0.4619],
          [ 0.7250,  0.1550]],

         [[ 2.3890, -2.0204],
          [ 0.6381, -2.4400],
          [-0.1989, -1.5296]],

         [[-0.0519,  0.9462],
          [ 0.2781,  0.6628],
          [-0.0373, -1.0839]],

         [[ 0.9965,  0.1387],
          [ 0.8618, -0.5206],
          [-0.8769,  0.3068]]]])


In [22]:
out = stacked.flatten(-2)
print('out形状：',out.shape)
print('out:\n',out)

out形状： torch.Size([1, 5, 6])
out:
 tensor([[[ 0.4473, -0.0838,  0.6370,  0.6283, -0.0917,  0.2281],
         [-0.0622, -0.5716,  1.7456, -0.4619,  0.7250,  0.1550],
         [ 2.3890, -2.0204,  0.6381, -2.4400, -0.1989, -1.5296],
         [-0.0519,  0.9462,  0.2781,  0.6628, -0.0373, -1.0839],
         [ 0.9965,  0.1387,  0.8618, -0.5206, -0.8769,  0.3068]]])
